# Day 03 - 线性回归：预测温度今天我们要做一件很酷的事情：**用历史数据预测未来的温度**！这就是机器学习中最基础的任务之一 —— **回归 (Regression)**。---## 今天的目标1. 理解什么是回归2. 用 scikit-learn 训练第一个模型3. 用模型预测温度并评估效果

## 1. 什么是回归？**回归**就是预测一个 **数值**（比如温度、价格、身高）。比如：- 预测明天的温度是 25°C → 回归- 预测明天会不会下雨 → 分类（不是回归）今天我们用 **线性回归 (Linear Regression)** 来预测温度。线性回归的思想很简单：找到一条 **最合适的直线** 来描述数据之间的关系。

In [ ]:
# 导入库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_squared_error, r2_scoreplt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]plt.rcParams["axes.unicode_minus"] = False# 读取数据df = pd.read_csv("data/weather_sample.csv", encoding="utf-8-sig")print(f"数据大小: {df.shape}")df.head()

## 2. 准备数据我们要用 **过去几天的温度** 来预测 **明天的温度**。首先，我们只看北京的数据，按日期排序。

In [ ]:
# 筛选北京的数据beijing = df[df["城市"] == "北京"].copy()beijing = beijing.sort_values("日期").reset_index(drop=True)print(f"北京数据量: {len(beijing)} 天")beijing.head()

In [ ]:
# 构造特征：用过去7天的温度预测第8天def create_features(data, window=7):    """用过去 window 天的温度预测下一天"""    X, y = [], []    temps = data["温度"].values    for i in range(window, len(temps)):        X.append(temps[i-window:i])  # 过去7天        y.append(temps[i])            # 第8天    return np.array(X), np.array(y)X, y = create_features(beijing, window=7)print(f"特征形状: {X.shape}")print(f"标签形状: {y.shape}")print(f"\n示例：过去7天温度 = {X[0]}")print(f"预测目标（第8天）= {y[0]:.1f}")

## 3. 划分训练集和测试集**训练集 (Training Set)**：用来教模型学习**测试集 (Test Set)**：用来检验模型学得好不好我们按时间顺序划分：前80%用来训练，后20%用来测试。

In [ ]:
# 按时间顺序划分（不能随机打乱，因为我们要预测未来）split = int(len(X) * 0.8)X_train, X_test = X[:split], X[split:]y_train, y_test = y[:split], y[split:]print(f"训练集: {len(X_train)} 个样本")print(f"测试集: {len(X_test)} 个样本")

## 4. 训练模型现在让 scikit-learn 来帮我们找到最合适的直线！

In [ ]:
# 创建并训练线性回归模型model = LinearRegression()model.fit(X_train, y_train)print("模型训练完成！")print(f"模型系数（每个过去温度的权重）: {model.coef_.round(3)}")print(f"模型截距: {model.intercept_:.3f}")

In [ ]:
# 在测试集上做预测y_pred = model.predict(X_test)# 对比预测值和真实值results = pd.DataFrame({    "真实温度": y_test,    "预测温度": y_pred.round(1),    "误差": (y_test - y_pred).round(1)})results.head(10)

In [ ]:
# 评估模型效果mse = mean_squared_error(y_test, y_pred)rmse = np.sqrt(mse)r2 = r2_score(y_test, y_pred)print(f"均方误差 (MSE): {mse:.2f}")print(f"均方根误差 (RMSE): {rmse:.2f}")print(f"R2 分数: {r2:.2f}")print(f"\nRMSE 解释：平均预测误差约 {rmse:.1f} 度")print(f"R2 解释：模型能解释 {r2*100:.0f}% 的温度变化")

In [ ]:
# 可视化：预测 vs 真实plt.figure(figsize=(12, 5))plt.subplot(1, 2, 1)plt.scatter(y_test, y_pred, alpha=0.6, color="steelblue")plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],         "r--", linewidth=2, label="完美预测线")plt.xlabel("真实温度 (C)", fontsize=12)plt.ylabel("预测温度 (C)", fontsize=12)plt.title("预测 vs 真实", fontsize=14)plt.legend()plt.subplot(1, 2, 2)plt.plot(y_test.values, label="真实温度", alpha=0.7)plt.plot(y_pred, label="预测温度", alpha=0.7)plt.xlabel("测试样本序号", fontsize=12)plt.ylabel("温度 (C)", fontsize=12)plt.title("温度变化对比", fontsize=14)plt.legend()plt.tight_layout()plt.show()

## 5. 你学到了什么？| 概念 | 说明 ||------|------|| 回归 | 预测一个数值 || 线性回归 | 用直线拟合数据 || 特征 (Feature) | 输入数据 || 标签 (Label) | 要预测的目标 || 训练集 | 用来教模型的数据 || 测试集 | 用来检验模型的数据 || RMSE | 预测误差的平均大小 || R2 分数 | 模型解释数据变化的能力（越接近1越好）|---## 挑战任务（选做）1. 试试用过去3天或5天来预测，效果有什么变化？2. 用上海的天气数据重新训练一个模型3. 把湿度和风速也加入特征，看看预测是否更准确